# Oura MCP — example analysis notebook

This notebook shows how to call `oura-mcp` tools directly from Python for local analysis.
It's meant as a starting point — copy it, swap in your own date ranges, and build on it.

## Setup

```bash
pip install oura-mcp
export OURA_PAT="your-token-here"        # or set ~/.oura-mcp/config.json
export OURA_MCP_CACHE_DIR=~/.oura-mcp/raw  # optional but recommended
```

In [ ]:
import asyncio
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from oura_mcp.server import mcp

_tools = {t.name: t for t in mcp._tool_manager.list_tools()}

async def call(name, **kwargs):
    """Convenience wrapper — calls a tool by name."""
    return await _tools[name].fn(**kwargs)

## 1. Check connectivity

In [ ]:
info = await call("oura_personal_info")
print(info)

## 2. Nightly summary table (last 30 days)

In [ ]:
result = await call(
    "oura_summary_table",
    start_date="2026-03-30",
    end_date="2026-04-28",
)

df = pd.DataFrame(result["data"])
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date").sort_index()
df.head()

## 3. Sleep stage composition chart

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

stages = [
    ("deep_min",  "#1f4e79", "Deep"),
    ("rem_min",   "#2e75b6", "REM"),
    ("light_min", "#9dc3e6", "Light"),
    ("awake_min", "#ffd966", "Awake"),
]

bottom = pd.Series(0, index=df.index)
for col, color, label in stages:
    vals = df[col].fillna(0)
    ax.bar(df.index, vals, bottom=bottom, color=color, label=label, width=0.8)
    bottom += vals

ax.set_xlabel("Date")
ax.set_ylabel("Minutes")
ax.set_title("Sleep stage composition — last 30 nights")
ax.legend(loc="upper right")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 4. HRV trend with 7-night rolling average

In [ ]:
hrv_result = await call(
    "oura_rolling_average",
    metric="average_hrv",
    start_date="2026-03-30",
    end_date="2026-04-28",
    window=7,
)

hrv_df = pd.DataFrame(hrv_result["data"])
hrv_df["date"] = pd.to_datetime(hrv_df["date"])
hrv_df = hrv_df.set_index("date")

fig, ax = plt.subplots(figsize=(14, 3))
ax.scatter(hrv_df.index, hrv_df["value"], s=20, alpha=0.5, color="steelblue", label="Nightly HRV")
ax.plot(hrv_df.index, hrv_df["rolling_avg"], color="steelblue", linewidth=2, label="7-night avg")
ax.set_ylabel("HRV (ms)")
ax.set_title("HRV trend")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 5. Deep sleep percentiles

In [ ]:
pct = await call(
    "oura_percentiles",
    metric="deep_sleep_duration",
    start_date="2026-03-30",
    end_date="2026-04-28",
    percentiles=[25, 50, 75, 90],
)

print(f"Nights analysed: {pct['count']}")
print(f"Mean:  {pct['mean']/60:.0f} min")
for p, v in pct["percentiles"].items():
    print(f"P{p:>3}: {v/60:.0f} min")

## 6. Hypnogram for a specific night

In [ ]:
hyp = await call("oura_render_hypnogram", date="2026-04-28")

print(f"Date:          {hyp.get('date')}")
print(f"Session type:  {hyp.get('session_type')}")
print(f"Total minutes: {hyp.get('total_minutes')}")
print(f"Hypnogram:     {hyp.get('hypnogram')}")
print(f"Key:           {hyp.get('key')}")

## 7. Score correlation: sleep vs readiness

In [ ]:
plot_df = df[["sleep_score", "readiness_score"]].dropna()

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(plot_df["sleep_score"], plot_df["readiness_score"], alpha=0.6, color="steelblue")
ax.set_xlabel("Sleep score")
ax.set_ylabel("Readiness score")
ax.set_title("Sleep vs readiness correlation")

corr = plot_df["sleep_score"].corr(plot_df["readiness_score"])
ax.text(0.05, 0.95, f"r = {corr:.2f}", transform=ax.transAxes, va="top")
plt.tight_layout()
plt.show()